In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:
# /data/beifen/zhongmin/slide-tag/代谢/scCellFie/2025_12_30/slide-tag_lung_scCellFie_metabolic_tasks.h5ad

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

H5AD_PATH = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/2026_3_24/slide-tag_lung_scCellFie_metabolic_tasks.h5ad"

SAMPLE_COL = "sample"

NORMAL_GROUPS = {"Conforming", "Mature"}
DEVIATING_GROUP = "Deviating"

def normalize_tls_group(x):
    if x is None:
        return "<NA>"
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "<na>", "none"}:
        return "<NA>"
    low = s.lower()
    if "conform" in low:
        return "Conforming"
    if "mature" in low:
        return "Mature"
    if "deviat" in low:
        return "Deviating"
    return s

def guess_tls_group_col(obs: pd.DataFrame) -> str:
    candidates = [
       
        "tls_degrow_id_fenlei",
       
    ]
    cols = list(obs.columns)

    best_col = None
    best_score = -1
    best_hit_set = set()

    # 先按候选名字优先
    ordered = [c for c in candidates if c in cols] + [c for c in cols if c not in candidates]

    for c in ordered:
        # 只对 object/category 列尝试，数值列跳过
        if pd.api.types.is_numeric_dtype(obs[c]):
            continue

        v = obs[c].astype(str).map(normalize_tls_group)
        hits = set(v.unique()) & {"Conforming", "Mature", "Deviating"}

        # 评分：命中类别越多越好，且要至少命中 2 类才算“像”
        score = len(hits)
        if score > best_score:
            best_score = score
            best_col = c
            best_hit_set = hits

        if score == 3:
            break

    if best_col is None or best_score < 2:
        # 给出可诊断信息
        preview = {}
        for c in cols[:30]:
            if not pd.api.types.is_numeric_dtype(obs[c]):
                u = pd.Series(obs[c].astype(str).map(normalize_tls_group).unique())
                preview[c] = u.head(10).tolist()
        raise KeyError(
            "无法自动识别 TLS 分组列（需要包含 Mature/Conforming/Deviating 中至少两类）。\n"
            f"请检查 adata.obs 里哪一列是 TLS 分组列，并把列名替换到代码里。\n"
            f"前30列的示例取值(每列最多10个)：{preview}"
        )

    print(f"[Info] TLS group column guessed: {best_col}  hits={sorted(best_hit_set)}")
    return best_col


# 读入数据
adata = sc.read_h5ad(H5AD_PATH)
print(adata)

# 基本检查
if SAMPLE_COL not in adata.obs.columns:
    raise KeyError(f"adata.obs 中不存在列 {SAMPLE_COL}，现有列：{list(adata.obs.columns)[:50]} ...")

# 自动猜 TLS 分组列
tls_group_col = guess_tls_group_col(adata.obs)

df = adata.obs[[SAMPLE_COL, tls_group_col]].copy()
df[SAMPLE_COL] = df[SAMPLE_COL].astype(str)
df["tls_group_norm"] = df[tls_group_col].map(normalize_tls_group)

# 只关心这三类，其余记为 <NA> 或其它值都不会算入 normal/deviating
df["is_normal_tls"] = df["tls_group_norm"].isin(NORMAL_GROUPS)
df["is_deviating_tls"] = (df["tls_group_norm"] == DEVIATING_GROUP)

# 每个 sample 是否包含正常/异常 TLS
by_sample = df.groupby(SAMPLE_COL, observed=True).agg(
    has_normal_tls=("is_normal_tls", "any"),
    has_deviating_tls=("is_deviating_tls", "any"),
    n_cells=("tls_group_norm", "size"),
    n_cells_normal=("is_normal_tls", "sum"),
    n_cells_deviating=("is_deviating_tls", "sum"),
)

by_sample["both_normal_and_deviating"] = by_sample["has_normal_tls"] & by_sample["has_deviating_tls"]

n_total_samples = by_sample.shape[0]
n_both = int(by_sample["both_normal_and_deviating"].sum())
n_only_normal = int((by_sample["has_normal_tls"] & ~by_sample["has_deviating_tls"]).sum())
n_only_deviating = int((~by_sample["has_normal_tls"] & by_sample["has_deviating_tls"]).sum())
n_neither = int((~by_sample["has_normal_tls"] & ~by_sample["has_deviating_tls"]).sum())

print("\n[Summary]")
print(f"Total samples: {n_total_samples}")
print(f"Samples with BOTH (Normal TLS + Deviating TLS): {n_both}")
print(f"Samples with ONLY Normal TLS: {n_only_normal}")
print(f"Samples with ONLY Deviating TLS: {n_only_deviating}")
print(f"Samples with NEITHER (no Mature/Conforming/Deviating detected): {n_neither}")

# 列出同时包含两类 TLS 的样本（示例前50个）
samples_both = by_sample.index[by_sample["both_normal_and_deviating"]].tolist()
print("\n[Examples] Sample IDs with BOTH (show up to 50):")
print(samples_both[:50])

# 如果你想看这些样本里 normal/deviating 细胞数（前20个）
print("\n[Top 20 samples with BOTH by total cells]")
print(
    by_sample.loc[samples_both]
            .sort_values("n_cells", ascending=False)
            .head(20)[["n_cells", "n_cells_normal", "n_cells_deviating"]]
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import mannwhitneyu

TLS_ID_COL = "tls_degrow_id_sample"
TLS_CLASS_COL = "tls_degrow_id_fenlei"
AROUND_TLSID_COL = "tls_degrow_id_around"

NORMAL_CLASSES = { "Mature","Conforming"}#"Mature",
DEVIATING_CLASS = "Deviating"

AGG_METHOD = "mean"          # "mean" / "sum" / "median"
MIN_TLS_PER_GROUP = 3        # 每组至少多少个 TLS 才比较

OUT_DIR = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/不分样本比较_伪bulk"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_TLS_PSEUDOBULK = os.path.join(OUT_DIR, f"TLS_pseudobulk_corePlusAround_{AGG_METHOD}.csv")
OUT_CSV = os.path.join(OUT_DIR, f"scCellFie_metabolic_tasks_Normal_vs_Deviating_TLS_pseudobulk_{AGG_METHOD}.csv")

def _norm_class(x):
    if x is None:
        return "<NA>"
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "<na>", "none"}:
        return "<NA>"
    low = s.lower()
    if "conform" in low:
        return "Conforming"
    if "mature" in low:
        return "Mature"
    if "deviat" in low:
        return "Deviating"
    return s

def _is_valid_id(x):
    if x is None:
        return False
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "<na>", "none"}:
        return False
    return True

def _around_to_core_id(x):
    if not _is_valid_id(x):
        return None
    s = str(x).strip()
    s2 = re.sub(r"_around.*$", "", s)
    return s2 if _is_valid_id(s2) else None

if "adata" not in globals():
    raise RuntimeError("未检测到变量 adata。请确认你已把 scCellFie metabolic_tasks 的 adata 读入内存。")

for c in [TLS_ID_COL, TLS_CLASS_COL, AROUND_TLSID_COL]:
    if c not in adata.obs.columns:
        raise KeyError(f"adata.obs 中缺少列 {c}")

task_names = adata.var_names.tolist()

obs = adata.obs[[TLS_ID_COL, TLS_CLASS_COL, AROUND_TLSID_COL]].copy()
obs[TLS_ID_COL] = obs[TLS_ID_COL].astype(str)
obs[AROUND_TLSID_COL] = obs[AROUND_TLSID_COL].astype(str)
obs["tls_class_norm"] = obs[TLS_CLASS_COL].map(_norm_class)
obs["around_core_id"] = obs[AROUND_TLSID_COL].map(_around_to_core_id)

core_mask = obs[TLS_ID_COL].map(_is_valid_id) & obs["tls_class_norm"].isin(NORMAL_CLASSES | {DEVIATING_CLASS})
core_df = obs.loc[core_mask, [TLS_ID_COL, "tls_class_norm"]].copy()
if core_df.shape[0] == 0:
    raise RuntimeError("没有检测到任何有效 TLS core 细胞（tls_degrow_id_sample + tls_degrow_id_fenlei）。")

tls2class = (
    core_df.groupby(TLS_ID_COL, observed=True)["tls_class_norm"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

normal_tls_ids = {k for k, v in tls2class.items() if v in NORMAL_CLASSES}
dev_tls_ids = {k for k, v in tls2class.items() if v == DEVIATING_CLASS}
selected_tls_ids = normal_tls_ids | dev_tls_ids

print("[Info] TLS summary (core-derived):")
print("  - normal TLS:", len(normal_tls_ids))
print("  - deviating TLS:", len(dev_tls_ids))
print("  - selected TLS total:", len(selected_tls_ids))

if len(normal_tls_ids) < MIN_TLS_PER_GROUP or len(dev_tls_ids) < MIN_TLS_PER_GROUP:
    raise RuntimeError(
        f"TLS 数量不足：normal={len(normal_tls_ids)}, dev={len(dev_tls_ids)}, "
        f"MIN_TLS_PER_GROUP={MIN_TLS_PER_GROUP}"
    )

keep_core = obs[TLS_ID_COL].isin(selected_tls_ids)
keep_around = obs["around_core_id"].isin(selected_tls_ids)

keep_any = keep_core | keep_around
if int(keep_any.sum()) == 0:
    raise RuntimeError("core+around 筛选后没有任何细胞被保留。请检查 around ID 是否能正确映射到 core TLS。")

obs_keep = obs.loc[keep_any, :].copy()

linked_tls = np.where(
    obs_keep[TLS_ID_COL].isin(selected_tls_ids),
    obs_keep[TLS_ID_COL].to_numpy(),
    obs_keep["around_core_id"].to_numpy()
)

linked_class = pd.Series(linked_tls).map(tls2class).fillna("<NA>").to_numpy()

group = np.full(shape=linked_class.shape[0], fill_value="<NA>", dtype=object)
group[np.isin(linked_class, list(NORMAL_CLASSES))] = "Normal"
group[linked_class == DEVIATING_CLASS] = "Deviating"

valid = np.isin(group, ["Normal", "Deviating"]) & pd.Series(linked_tls).map(_is_valid_id).to_numpy()
obs_keep = obs_keep.iloc[valid, :].copy()
group = group[valid]
linked_tls = linked_tls[valid]

is_core_used = obs_keep[TLS_ID_COL].isin(selected_tls_ids).to_numpy()
is_around_used = (~is_core_used) & obs_keep["around_core_id"].isin(selected_tls_ids).to_numpy()

print("[Info] Kept cells (after labeling):")
print("  - core used  :", int(is_core_used.sum()))
print("  - around used:", int(is_around_used.sum()))
print("  - total used :", obs_keep.shape[0])

X = adata[obs_keep.index, :].X
if issparse(X):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

tls_codes, tls_uniques = pd.factorize(pd.Series(linked_tls), sort=True)
tls_uniques = tls_uniques.astype(str)
n_tls = len(tls_uniques)
counts = np.bincount(tls_codes, minlength=n_tls).astype(np.int64)

tls_group = pd.Series(tls_uniques).map(lambda x: "Normal" if x in normal_tls_ids else ("Deviating" if x in dev_tls_ids else "<NA>")).to_numpy()

if AGG_METHOD in {"mean", "sum"}:
    pb = np.zeros((n_tls, X.shape[1]), dtype=np.float32)
    for j in range(X.shape[1]):
        s = np.bincount(tls_codes, weights=X[:, j].astype(np.float64), minlength=n_tls)
        if AGG_METHOD == "mean":
            denom = np.maximum(counts, 1)
            pb[:, j] = (s / denom).astype(np.float32)
        else:
            pb[:, j] = s.astype(np.float32)
elif AGG_METHOD == "median":
    pb = np.zeros((n_tls, X.shape[1]), dtype=np.float32)
    for k in range(n_tls):
        idx = np.where(tls_codes == k)[0]
        if idx.size == 0:
            pb[k, :] = np.nan
        else:
            pb[k, :] = np.nanmedian(X[idx, :], axis=0).astype(np.float32)
else:
    raise ValueError("AGG_METHOD 仅支持 'mean' / 'sum' / 'median'")

pb_df = pd.DataFrame(pb, columns=task_names)
pb_df.insert(0, "tls_id", tls_uniques)
pb_df.insert(1, "tls_group", tls_group)
pb_df.insert(2, "n_cells_in_tls_used(core+around)", counts)

pb_df.to_csv(OUT_TLS_PSEUDOBULK, index=False)
print("[Saved]", OUT_TLS_PSEUDOBULK, " rows=", pb_df.shape[0], " cols=", pb_df.shape[1])

mask_tls_norm = (pb_df["tls_group"].to_numpy() == "Normal")
mask_tls_dev = (pb_df["tls_group"].to_numpy() == "Deviating")

n_tls_norm = int(mask_tls_norm.sum())
n_tls_dev = int(mask_tls_dev.sum())

print("[Info] TLS counts used for test:")
print("  - Normal TLS   :", n_tls_norm)
print("  - Deviating TLS:", n_tls_dev)

if n_tls_norm < MIN_TLS_PER_GROUP or n_tls_dev < MIN_TLS_PER_GROUP:
    raise RuntimeError(
        f"TLS 数量不足：Normal={n_tls_norm}, Deviating={n_tls_dev}, MIN_TLS_PER_GROUP={MIN_TLS_PER_GROUP}"
    )

rows = []
for j, tname in enumerate(task_names):
    x_norm = pb[mask_tls_norm, j]
    x_dev = pb[mask_tls_dev, j]

    x_norm = x_norm[np.isfinite(x_norm)]
    x_dev = x_dev[np.isfinite(x_dev)]

    if x_norm.size < MIN_TLS_PER_GROUP or x_dev.size < MIN_TLS_PER_GROUP:
        continue

    med_norm = float(np.median(x_norm))
    med_dev = float(np.median(x_dev))
    delta_med = med_dev - med_norm

    if (np.nanstd(x_norm) == 0) and (np.nanstd(x_dev) == 0) and (med_norm == med_dev):
        p = 1.0
        u = (x_dev.size * x_norm.size) / 2.0
    else:
        u, p = mannwhitneyu(x_dev, x_norm, alternative="two-sided", method="asymptotic")

    n1 = float(x_dev.size)
    n2 = float(x_norm.size)
    rbc = (2.0 * u / (n1 * n2)) - 1.0

    rows.append({
        "task": tname,
        "agg_method": AGG_METHOD,
        "n_tls_normal": int(x_norm.size),
        "n_tls_deviating": int(x_dev.size),
        "median_tls_normal": med_norm,
        "median_tls_deviating": med_dev,
        "delta_median_tls(dev-normal)": delta_med,
        "u_stat": float(u),
        "p_value": float(p),
        "rank_biserial": float(rbc),
        "n_cells_total_in_normal_tls_used": int(pb_df.loc[mask_tls_norm, "n_cells_in_tls_used(core+around)"].sum()),
        "n_cells_total_in_deviating_tls_used": int(pb_df.loc[mask_tls_dev, "n_cells_in_tls_used(core+around)"].sum()),
        "n_tls_normal_total(core-derived)": int(len(normal_tls_ids)),
        "n_tls_deviating_total(core-derived)": int(len(dev_tls_ids)),
    })

if len(rows) == 0:
    raise RuntimeError("没有得到任何可用比较结果（可能 TLS 数量太少或数据全为 NaN/常数）。")

df_all = pd.DataFrame(rows)
df_all = df_all.sort_values(["p_value", "task"], ascending=[True, True]).reset_index(drop=True)

df_all.to_csv(OUT_CSV, index=False)
print("\n[Saved]", OUT_CSV)
print("[Info] Total tasks compared:", df_all.shape[0])
print("[Info] Columns:", list(df_all.columns))


In [ ]:
import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from adjustText import adjust_text

# =========================
# 0) 输入/输出
# =========================
IN_CSV = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/不分样本比较_伪bulk/scCellFie_metabolic_tasks_Normal_vs_Deviating_TLS_pseudobulk_mean.csv"

OUT_DIR = os.path.dirname(IN_CSV)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PNG = os.path.join(OUT_DIR, "Volcano_TLS_pseudobulk_5modules_colored.png")
OUT_PDF = os.path.join(OUT_DIR, "Volcano_TLS_pseudobulk_5modules_colored.pdf")

# =========================
# 1) 生物学筛选后的主通路 + 颜色
# =========================
PV_CUTOFF = 0.05

MODULE_INFO = {
    "M1: PIP2→IP3 / Ca²⁺ signal": "#e41a1c",
    "M2: Glycan maturation ↓": "#377eb8",
    "M3: Arginine-Gln axis ↑": "#ff7f00",
    "M4: Lipid / Cardiolipin ↑": "#984ea3",
}

TASK_TO_MODULE = {
    "Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate to 1D-myo-inositol 1,4,5-trisphosphate": "M1: PIP2→IP3 / Ca²⁺ signal",
    "Phosphatidyl-inositol synthesis": "M1: PIP2→IP3 / Ca²⁺ signal",
    "Sialylation (addition of sialic acid)": "M2: Glycan maturation ↓",
    "Branching (N-acetylglucosaminyltransferases)": "M2: Glycan maturation ↓",
    "Fucosylation (addition of fucose)": "M2: Glycan maturation ↓",
    "Arginine synthesis": "M3: Arginine-Gln axis ↑",
    "Conversion of aspartate to arginine": "M3: Arginine-Gln axis ↑",
    "Arginine degradation": "M3: Arginine-Gln axis ↑",
    "Cardiolipin synthesis": "M4: Lipid / Cardiolipin ↑",
    "Triacylglycerol synthesis": "M4: Lipid / Cardiolipin ↑",
    "Synthesis of palmitoyl-CoA": "M4: Lipid / Cardiolipin ↑",
    "Phosphatidyl-serine synthesis": "M4: Lipid / Cardiolipin ↑",
    "Synthesis of glucocerebroside": "M4: Lipid / Cardiolipin ↑",
}

TASKS_TO_LABEL = list(TASK_TO_MODULE.keys())
LABEL_ONLY_SIG = True
CAPTION_WRAP_WIDTH = 120
CAPTION_MAX_LINES = 120

# =========================
# 2) 工具函数
# =========================
def _clean_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def _normalize_task_key(s: str) -> str:
    return _clean_spaces(s).lower()

def _find_delta_col(df: pd.DataFrame) -> str:
    candidates = [
        "delta_median_tls(dev-normal)", "delta_median(dev-normal)", "delta_median",
        "delta_mean_tls(dev-normal)", "delta_mean(dev-normal)", "delta_mean", "delta",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    cols = list(df.columns)
    for c in cols:
        lc = c.lower()
        if "delta" in lc and ("dev" in lc or "deviat" in lc):
            return c
    for c in cols:
        if "delta" in c.lower():
            return c
    raise KeyError(f"未找到 delta 列。当前列：{cols}")

# =========================
# 3) 读入数据
# =========================
df = pd.read_csv(IN_CSV)
for c in ["task", "p_value"]:
    if c not in df.columns:
        raise KeyError(f"缺少必要列：{c}。当前列：{list(df.columns)}")

delta_col = _find_delta_col(df)
df["task"] = df["task"].astype(str).map(_clean_spaces)
df["p_value"] = pd.to_numeric(df["p_value"], errors="coerce")
df["delta"] = pd.to_numeric(df[delta_col], errors="coerce")
df = df.dropna(subset=["task", "p_value", "delta"]).copy()

p_safe = df["p_value"].clip(lower=1e-300)
df["neglog10p"] = -np.log10(p_safe)
df["sig"] = df["p_value"] < PV_CUTOFF

# =========================
# 4) 过滤出要标注的通路
# =========================
task_to_module_norm = {_normalize_task_key(k): v for k, v in TASK_TO_MODULE.items()}
want_keys = {_normalize_task_key(t): t for t in TASKS_TO_LABEL}
df["_task_key"] = df["task"].map(_normalize_task_key)
df_want = df[df["_task_key"].isin(want_keys.keys())].copy()

found_keys = set(df_want["_task_key"].unique().tolist())
missing = [t for k, t in want_keys.items() if k not in found_keys]
if missing:
    print("[WARN] Missing tasks in volcano input:")
    for t in missing:
        print("  -", t)

if LABEL_ONLY_SIG:
    df_want = df_want[df_want["sig"]].copy()

df_want = df_want.sort_values(["task", "p_value"]).drop_duplicates(subset=["task"], keep="first").copy()
df_want["module"] = df_want["_task_key"].map(task_to_module_norm)
to_label = df_want.copy()


In [ ]:
# Volcano caption split layout (keep original plot style)
import textwrap

required_vars = [
    'df', 'to_label', 'MODULE_INFO', 'PV_CUTOFF', 'OUT_PNG', 'OUT_PDF',
    '_clean_spaces', 'IN_CSV', 'adjust_text', 'TASKS_TO_LABEL', 'TASK_TO_MODULE'
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"请先运行上一格火山图主代码，缺少变量: {missing_vars}")

# 重新编号：按照 M1 -> M5 的任务顺序编号，而不是按左右两侧/效应值大小编号
_task_rank = {_clean_spaces(task): i for i, task in enumerate(TASKS_TO_LABEL)}
plot_df = to_label.copy()
plot_df['task_rank'] = plot_df['task'].map(lambda x: _task_rank.get(_clean_spaces(x), 10**6))
plot_df = plot_df.sort_values(['task_rank', 'p_value', 'delta'], ascending=[True, True, False]).reset_index(drop=True)
plot_df['label_id'] = range(1, len(plot_df) + 1)


def _build_group_caption(df_group, header, wrap_width, max_lines):
    lines = [header]
    if df_group.empty:
        lines.append('None')
    else:
        for r in df_group.sort_values('label_id').itertuples(index=False):
            entry = f"[{int(r.label_id)}] [{r.module}]  {_clean_spaces(r.task)}"
            wrapped = textwrap.wrap(entry, width=wrap_width) or ['']
            if len(lines) + len(wrapped) > max_lines:
                lines.append('… (caption truncated)')
                break
            lines.extend(wrapped)
    return '\n'.join(lines), len(lines)


df_pos_cap = plot_df[plot_df['delta'] > 0].copy().sort_values(['label_id'])
df_neg_cap = plot_df[plot_df['delta'] < 0].copy().sort_values(['label_id'])
left_caption_text, left_lines = _build_group_caption(df_neg_cap, 'Normal higher (Δ < 0):', wrap_width=50, max_lines=36)
right_caption_text, right_lines = _build_group_caption(df_pos_cap, 'Deviating higher (Δ > 0):', wrap_width=78, max_lines=60)

# Illustrator-friendly text output + larger fonts
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

TITLE_SIZE = 15
AXIS_LABEL_SIZE = 13.5
TICK_LABEL_SIZE = 11.5
LEGEND_FONT_SIZE = 11
NUMBER_FONT_SIZE = 10.5
CAPTION_FONT_SIZE = 9.8

caption_lines = max(left_lines, right_lines)
fig_h = 7.2
fig, ax = plt.subplots(figsize=(13.5, fig_h), dpi=170)
bottom_margin = 0.11
fig.subplots_adjust(right=0.76, bottom=bottom_margin)

bg = df[~df['sig']]
ax.scatter(bg['delta'], bg['neglog10p'], s=14, alpha=0.28, linewidths=0, color='#aaaaaa', label='Not sig')

labeled_tasks = set(plot_df['_task_key'].tolist())
sig_unlabeled_pos = df[(df['sig']) & (df['delta'] > 0) & (~df['_task_key'].isin(labeled_tasks))]
sig_unlabeled_neg = df[(df['sig']) & (df['delta'] < 0) & (~df['_task_key'].isin(labeled_tasks))]
ax.scatter(sig_unlabeled_pos['delta'], sig_unlabeled_pos['neglog10p'], s=22, alpha=0.70, linewidths=0, color='#fc8d62', label='Sig Deviating↑')
ax.scatter(sig_unlabeled_neg['delta'], sig_unlabeled_neg['neglog10p'], s=22, alpha=0.70, linewidths=0, color='#66c2a5', label='Sig Normal↑')

for mod, color in MODULE_INFO.items():
    sub = plot_df[plot_df['module'] == mod]
    if len(sub) == 0:
        continue
    ax.scatter(sub['delta'], sub['neglog10p'], s=55, alpha=0.95, linewidths=0.4,
               edgecolors='black', color=color, zorder=5)

ax.axhline(-np.log10(PV_CUTOFF), linestyle='--', linewidth=1, color='#555555', alpha=0.7)
ax.axvline(0, linestyle='--', linewidth=1, color='#555555', alpha=0.7)

xq = np.quantile(df['delta'].values, [0.01, 0.99])
xspan = float(xq[1] - xq[0]) if np.isfinite(xq).all() else 1.0
xpad = max(0.08, 0.22 * xspan)
ax.set_xlim(float(xq[0] - xpad), float(xq[1] + xpad))

ymax = float(np.quantile(df['neglog10p'].values, 0.995))
ax.set_ylim(0, max(5.0, ymax * 1.15))

scope_title = '(TLS pseudo-bulk, core + around)'
if '不用aound' in IN_CSV or '不用around' in IN_CSV or 'coreOnly' in OUT_PNG:
    scope_title = '(TLS pseudo-bulk, core)'
ax.set_title('Volcano: scCellFie metabolic tasks — Normal TLS vs Deviating TLS\n' + scope_title, fontsize=TITLE_SIZE, pad=10)
ax.set_xlabel('Δ median (Deviating − Normal)', fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel('−log₁₀(p)  [Mann–Whitney U]', fontsize=AXIS_LABEL_SIZE)

legend_handles = [
    mpatches.Patch(color='#aaaaaa', label='Not sig'),
    mpatches.Patch(color='#fc8d62', label='Sig Deviating↑ (unlabeled)'),
    mpatches.Patch(color='#66c2a5', label='Sig Normal↑  (unlabeled)'),
]
for mod, color in MODULE_INFO.items():
    legend_handles.append(mpatches.Patch(color=color, label=mod))
ax.legend(handles=legend_handles, frameon=False, loc='upper left',
          bbox_to_anchor=(1.01, 1.00), fontsize=LEGEND_FONT_SIZE)
ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.30)
ax.tick_params(axis='both', labelsize=TICK_LABEL_SIZE)

xlim = ax.get_xlim()
ylim = ax.get_ylim()
xrange = xlim[1] - xlim[0]
yrange = ylim[1] - ylim[0]
EDGE_MARGIN = 0.015

texts_inside = []
x_inside = []
y_inside = []

for row in plot_df.sort_values('label_id').itertuples(index=False):
    x_real = float(row.delta)
    y = float(row.neglog10p)
    lab = str(int(row.label_id))
    mod_color = MODULE_INFO.get(row.module, '#333333')

    if x_real < xlim[0] or x_real > xlim[1]:
        if x_real < xlim[0]:
            x_edge = xlim[0] + EDGE_MARGIN * xrange
        else:
            x_edge = xlim[1] - EDGE_MARGIN * xrange
        y_clamped = min(max(y, ylim[0] + 0.03 * yrange), ylim[1] - 0.03 * yrange)

        ax.scatter([x_edge], [y_clamped], s=55, alpha=0.95, linewidths=0.4,
                   edgecolors='black', color=mod_color, zorder=5, clip_on=False)

        side = 1 if x_real < xlim[0] else -1
        x_text = x_edge + side * 0.025 * xrange

        ax.annotate(
            lab,
            xy=(x_edge, y_clamped),
            xytext=(x_text, y_clamped),
            textcoords='data',
            ha='center', va='center',
            fontsize=NUMBER_FONT_SIZE, fontweight='bold',
            color='white',
            bbox=dict(boxstyle='circle,pad=0.22', fc=mod_color, ec='black', lw=0.55, alpha=0.93),
            arrowprops=dict(arrowstyle='->', lw=0.9, alpha=0.75, color=mod_color,
                            shrinkA=7, shrinkB=4),
            annotation_clip=False, zorder=10,
        )
    else:
        t = ax.text(
            x_real, y, lab,
            ha='center', va='center',
            fontsize=NUMBER_FONT_SIZE, fontweight='bold',
            color='white',
            bbox=dict(boxstyle='circle,pad=0.22', fc=mod_color, ec='black', lw=0.55, alpha=0.93),
            zorder=10,
        )
        texts_inside.append(t)
        x_inside.append(x_real)
        y_inside.append(y)

if len(texts_inside) > 1:
    adjust_text(
        texts_inside, x=x_inside, y=y_inside, ax=ax,
        arrowprops=dict(arrowstyle='->', lw=0.9, alpha=0.75, color='#555555',
                        shrinkA=0, shrinkB=4),
        expand=(2.0, 2.0),
        force_text=(1.5, 1.5),
        force_points=(1.2, 1.2),
        max_move=None,
        only_move='xy',
        ensure_inside_axes=True,
    )

# The long ID-to-pathway explanation is exported as a separate graphical key below.

plt.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
plt.savefig(OUT_PDF, bbox_inches='tight')
plt.close(fig)

print('[Saved refined caption split]')
print('  -', OUT_PNG)
print('  -', OUT_PDF)
print('[Info] Label order by tasks:', plot_df[['label_id', 'module', 'task']].to_dict('records'))




In [ ]:
# Volcano pathway number key (separate graphical legend)
import os
import math
import textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

required_vars = ['plot_df', 'MODULE_INFO', 'OUT_DIR', '_clean_spaces']
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f'请先运行上一格火山图代码，缺少变量: {missing_vars}')

KEY_OUT_PNG = os.path.join(OUT_DIR, 'Volcano_TLS_pseudobulk_5modules_pathway_key.png')
KEY_OUT_PDF = os.path.join(OUT_DIR, 'Volcano_TLS_pseudobulk_5modules_pathway_key.pdf')

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

KEY_DIRECTION_COLORS = {
    'Normal higher': '#377eb8',
    'Deviating higher': '#e41a1c',
}
COLOR_TEXT = '#1f2937'
COLOR_MUTED = '#6b7280'
COLOR_LINE = '#d1d5db'

key_df = plot_df[['label_id', 'module', 'task', 'delta', 'p_value']].copy()
key_df['direction'] = np.where(key_df['delta'] > 0, 'Deviating higher', 'Normal higher')
module_order = [m for m in MODULE_INFO.keys() if m in set(key_df['module'])]
module_order += [m for m in key_df['module'].dropna().unique().tolist() if m not in module_order]

MODULE_TITLE_ALIAS = {
    'M1: PIP2→IP3 / Ca²⁺ signal': 'M1  PIP2-IP3 / Ca2+ signal',
    'M2: Glycan maturation ↓': 'M2  Glycan maturation',
    'M3: Arginine-Gln axis ↑': 'M3  Arginine-Gln axis',
    'M4: Lipid / Cardiolipin ↑': 'M4  Lipid / Cardiolipin',
}


def _wrap_task_name(task, width=34):
    return '\n'.join(textwrap.wrap(_clean_spaces(task), width=width, break_long_words=False))


def _direction_label(direction):
    direction = str(direction)
    if direction.startswith('Normal'):
        return 'Normal'
    if direction.startswith('Deviating'):
        return 'Deviating'
    return direction.replace(' higher', '')


n_cols = 2
n_rows = int(math.ceil(max(len(module_order), 1) / n_cols))
fig_w = 12.7
fig_h = 2.75 * n_rows + 1.0
fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), dpi=170)
axes = np.asarray(axes).reshape(n_rows, n_cols)
fig.patch.set_facecolor('white')

for ax in axes.ravel():
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

for ax, module in zip(axes.ravel(), module_order):
    mod_color = MODULE_INFO.get(module, '#6b7280')
    sub = key_df[key_df['module'] == module].sort_values('label_id').reset_index(drop=True)
    n = len(sub)
    title = MODULE_TITLE_ALIAS.get(module, module)

    # Module header: a compact colored rail instead of a full text-heavy caption.
    ax.text(0.02, 0.94, title, ha='left', va='center',
            fontsize=16.5, fontweight='bold', color=mod_color)
    ax.text(0.98, 0.94, f'{n} pathways', ha='right', va='center',
            fontsize=12.0, color=COLOR_MUTED)
    ax.plot([0.02, 0.98], [0.875, 0.875], color=mod_color, lw=2.0,
            alpha=0.85, solid_capstyle='round')

    if n == 0:
        ax.text(0.5, 0.48, 'No labeled pathways', ha='center', va='center',
                fontsize=12.5, color=COLOR_MUTED)
        continue

    y_positions = np.linspace(0.74, 0.12, n) if n > 1 else np.array([0.50])
    rail_x = 0.055
    num_x = 0.115
    text_x = 0.185
    badge_x = 0.965

    ax.plot([rail_x, rail_x], [y_positions.min() - 0.055, y_positions.max() + 0.055],
            color=mod_color, lw=0.92, alpha=0.58, solid_capstyle='round')

    for row, y in zip(sub.itertuples(index=False), y_positions):
        direction_color = KEY_DIRECTION_COLORS[row.direction]
        ax.plot([rail_x, num_x - 0.035], [y, y], color=mod_color,
                lw=0.58, alpha=0.70, solid_capstyle='round')
        ax.scatter([num_x], [y], s=390, color=mod_color, alpha=0.96,
                   edgecolors='#202020', linewidths=0.45, zorder=4)
        ax.text(num_x, y, str(int(row.label_id)), ha='center', va='center',
                fontsize=13.5, fontweight='bold', color='white', zorder=5)

        ax.text(text_x, y, _wrap_task_name(row.task), ha='left', va='center',
                fontsize=14.0, color=COLOR_TEXT, linespacing=1.05)
        ax.text(badge_x, y, _direction_label(row.direction), ha='right', va='center',
                fontsize=13.0, fontweight='bold', color=direction_color,
                bbox=dict(boxstyle='round,pad=0.14', fc='white', ec=direction_color,
                          lw=0.1, alpha=0.96))

# If the grid has empty panels, keep them blank.
for ax in axes.ravel()[len(module_order):]:
    ax.axis('off')

fig.suptitle('Pathway key for numbered metabolic tasks in the volcano plot',
             fontsize=21.0, fontweight='bold', color=COLOR_TEXT, y=0.978)
fig.text(0.5, 0.035,
         'Numbers match the volcano labels. Colored rails group tasks by metabolic module; right badges indicate which TLS group has higher pathway activity.',
         ha='center', va='bottom', fontsize=12.5, color=COLOR_MUTED)
fig.subplots_adjust(left=0.045, right=0.985, top=0.86, bottom=0.085,
                    wspace=0.10, hspace=0.16)

fig.savefig(KEY_OUT_PNG, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(KEY_OUT_PDF, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)

print('[Saved pathway key]')
print('  -', KEY_OUT_PNG)
print('  -', KEY_OUT_PDF)

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
from scipy.sparse import issparse
from scipy.stats import mannwhitneyu, spearmanr
import matplotlib; import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import defaultdict
warnings.filterwarnings("ignore", category=FutureWarning)

matplotlib.rcParams.update({
    'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],
    'font.size':15,'axes.titlesize':18,'axes.labelsize':15,
    'xtick.labelsize':13,'ytick.labelsize':13,'legend.fontsize':13,
    'figure.dpi':150,'savefig.dpi':300,'savefig.bbox':'tight','pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none',
    'axes.linewidth':1.0,'axes.spines.top':False,'axes.spines.right':False,
})

OUT_BASE = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/不分样本比较_伪bulk/验证"
os.makedirs(OUT_BASE, exist_ok=True)

SAMPLE_COL, TLS_ID_COL = "sample", "tls_degrow_id_sample"
TLS_CLASS_COL, AROUND_COL = "tls_degrow_id_fenlei", "tls_degrow_id_around"
NORMAL_CLASSES = {"Mature","Conforming"}
DEVIATING_CLASS = "Deviating"

# 只保留更可能真实影响 TLS 发育的主通路
MODULES = {
    "M1: PIP2-IP3 Ca2+": [
        "Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate to 1D-myo-inositol 1,4,5-trisphosphate",
        "Phosphatidyl-inositol synthesis"],
    "M2: Glycan maturation": [
        "Sialylation (addition of sialic acid)",
        "Branching (N-acetylglucosaminyltransferases)",
        "Fucosylation (addition of fucose)"],
    "M3: Arg-Gln axis": [
        "Arginine synthesis",
        "Conversion of aspartate to arginine",
        "Arginine degradation"],
    "M4: Lipid-Cardiolipin": [
        "Cardiolipin synthesis",
        "Triacylglycerol synthesis",
        "Synthesis of palmitoyl-CoA",
        "Phosphatidyl-serine synthesis",
        "Synthesis of glucocerebroside"],
}
MODULE_COLORS = {
    "M1: PIP2-IP3 Ca2+":"#e41a1c",
    "M2: Glycan maturation":"#377eb8",
    "M3: Arg-Gln axis":"#ff7f00",
    "M4: Lipid-Cardiolipin":"#984ea3",
}
BIOLOGICALLY_PRIORITIZED_TASKS = [
    "Conversion of 1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate to 1D-myo-inositol 1,4,5-trisphosphate",
    "Phosphatidyl-inositol synthesis",
    "Sialylation (addition of sialic acid)",
    "Branching (N-acetylglucosaminyltransferases)",
    "Fucosylation (addition of fucose)",
    "Arginine synthesis",
    "Conversion of aspartate to arginine",
    "Arginine degradation",
    "Cardiolipin synthesis",
    "Triacylglycerol synthesis",
    "Synthesis of palmitoyl-CoA",
    "Phosphatidyl-serine synthesis",
    "Synthesis of glucocerebroside",
]

ALL_MODULE_TASKS, TASK_TO_MODULE = [], {}
for m, ts in MODULES.items():
    for t in ts:
        ALL_MODULE_TASKS.append(t)
        TASK_TO_MODULE[t] = m
KEY_TASKS = ALL_MODULE_TASKS.copy()

def _norm_class(x):
    if x is None: return "<NA>"
    s=str(x).strip()
    if s=="" or s.lower() in {"nan","<na>","none"}: return "<NA>"
    low=s.lower()
    if "conform" in low: return "Conforming"
    if "mature" in low: return "Mature"
    if "deviat" in low: return "Deviating"
    return s
def _is_valid_id(x):
    if x is None: return False
    s=str(x).strip(); return s!="" and s.lower() not in {"nan","<na>","none"}
def _around_to_core_id(x):
    if not _is_valid_id(x): return None
    s=re.sub(r"_around.*$","",str(x).strip()); return s if _is_valid_id(s) else None
def _get_tls_group(c):
    if c in NORMAL_CLASSES: return "Normal"
    if c==DEVIATING_CLASS: return "Deviating"
    return "<NA>"
def _short_task(t,m=35):
    return t if len(t)<=m else t[:m-3]+"..."
def _task_key_for_label(t):
    return _clean_spaces(t) if '_clean_spaces' in globals() else re.sub(r"\s+", " ", str(t)).strip()
def _refresh_task_label_ids():
    label_source = globals().get('TASKS_TO_LABEL', BIOLOGICALLY_PRIORITIZED_TASKS)
    task_to_label_id = {_task_key_for_label(t): i + 1 for i, t in enumerate(label_source)}
    if 'plot_df' in globals() and {'task', 'label_id'}.issubset(set(plot_df.columns)):
        task_to_label_id.update({_task_key_for_label(r.task): int(r.label_id) for r in plot_df.itertuples(index=False)})
    return task_to_label_id
TASK_TO_LABEL_ID = _refresh_task_label_ids()
def _task_number_label(t):
    label_id = TASK_TO_LABEL_ID.get(_task_key_for_label(t))
    return str(label_id) if label_id is not None else _short_task(str(t), 45)

task_names = adata.var_names.tolist()
for t in ALL_MODULE_TASKS: assert t in task_names
print(f"[OK] biologically prioritized tasks: {len(ALL_MODULE_TASKS)}")


### 分析1：按样本一致性（热图 + 统计检验星号 + 方向一致率）

In [ ]:
obs_full = adata.obs[[SAMPLE_COL,TLS_ID_COL,TLS_CLASS_COL,AROUND_COL]].copy()
for c in [SAMPLE_COL,TLS_ID_COL,AROUND_COL]: obs_full[c]=obs_full[c].astype(str)
obs_full["cls"]=obs_full[TLS_CLASS_COL].map(_norm_class)
obs_full["around_cid"]=obs_full[AROUND_COL].map(_around_to_core_id)

core_mask=obs_full[TLS_ID_COL].map(_is_valid_id)&obs_full["cls"].isin(NORMAL_CLASSES|{DEVIATING_CLASS})
tls2class=obs_full.loc[core_mask].groupby(TLS_ID_COL,observed=True)["cls"].agg(lambda x:x.value_counts().index[0]).to_dict()
tls2group={k:_get_tls_group(v) for k,v in tls2class.items()}
tls2sample=obs_full.loc[core_mask].groupby(TLS_ID_COL,observed=True)[SAMPLE_COL].first().to_dict()

sample_tls=defaultdict(lambda:{"Normal":set(),"Deviating":set()})
for tid,grp in tls2group.items():
    if grp in ("Normal","Deviating"):
        smp=tls2sample.get(tid)
        if smp: sample_tls[smp][grp].add(tid)
valid_samples=sorted([s for s,d in sample_tls.items() if len(d["Normal"])>=1 and len(d["Deviating"])>=1])
print(f"[Info] {len(valid_samples)} samples with both groups")

X_full=adata.X
if issparse(X_full): X_full=X_full.toarray()
X_full=np.asarray(X_full,dtype=np.float32)
focus_idx={t:task_names.index(t) for t in ALL_MODULE_TASKS}

rows=[]
for smp in valid_samples:
    sm=obs_full[SAMPLE_COL].values==smp; so=obs_full.loc[sm]
    stls=sample_tls[smp]["Normal"]|sample_tls[smp]["Deviating"]
    keep=so[TLS_ID_COL].isin(stls)|so["around_cid"].isin(stls)
    if keep.sum()==0: continue
    linked=np.where(so.loc[keep,TLS_ID_COL].isin(stls).values,so.loc[keep,TLS_ID_COL].values,so.loc[keep,"around_cid"].values)
    groups=np.array([tls2group.get(t,"<NA>") for t in linked])
    valid=np.isin(groups,["Normal","Deviating"])
    cidx=np.where(sm)[0][np.where(keep.values)[0]][valid]; groups=groups[valid]
    if (groups=="Normal").sum()<10 or (groups=="Deviating").sum()<10: continue
    mn,md_mask=groups=="Normal",groups=="Deviating"
    for t in ALL_MODULE_TASKS:
        j=focus_idx[t]; xn,xd=X_full[cidx[mn],j],X_full[cidx[md_mask],j]
        xn,xd=xn[np.isfinite(xn)],xd[np.isfinite(xd)]
        if xn.size<10 or xd.size<10: continue
        try:
            u,p=mannwhitneyu(xd,xn,alternative="two-sided",method="asymptotic"); rbc=(2*u/(xd.size*xn.size))-1
        except:
            rbc,p=0,1
        rows.append({"sample":smp,"task":t,"module":TASK_TO_MODULE[t],"rbc":rbc,"p_value":p,"direction":"Dev↑" if rbc>0 else "Normal↑"})
df_ps=pd.DataFrame(rows)

pooled=pd.read_csv("/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/不分样本比较_伪bulk/scCellFie_metabolic_tasks_Normal_vs_Deviating_TLS_pseudobulk_mean.csv")
exp_dir={r["task"]:("Dev↑" if r["delta_median_tls(dev-normal)"]>0 else "Normal↑") for _,r in pooled.iterrows()}
con_rows=[]
for t in ALL_MODULE_TASKS:
    sub=df_ps[df_ps["task"]==t]
    if len(sub)==0: continue
    sub_sig=sub[sub["p_value"]<0.05]
    if len(sub_sig)==0: continue
    exp=exp_dir.get(t,"Dev↑"); nc=int((sub_sig["direction"]==exp).sum())
    con_rows.append({
        "task":t,
        "module":TASK_TO_MODULE[t],
        "n_samples_total":len(sub),
        "n_samples_sig":len(sub_sig),
        "n_consistent":nc,
        "consistency_rate":nc/len(sub_sig)
    })
df_con=pd.DataFrame(con_rows)
df_ps.to_csv(os.path.join(OUT_BASE,"1_per_sample_effect_sizes.csv"),index=False)
df_con.to_csv(os.path.join(OUT_BASE,"1_consistency_summary.csv"),index=False)

OTHER_CON = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/不分样本比较_伪bulk_不用aound/验证/1_consistency_summary.csv"
here_keep=set(df_con.loc[df_con["consistency_rate"]>0.60,"task"]) if len(df_con)>0 else set()
if os.path.exists(OTHER_CON):
    other_con=pd.read_csv(OTHER_CON)
    other_keep=set(other_con.loc[other_con["consistency_rate"]>0.60,"task"])
    TRUE_DIFF_TASKS=[t for t in BIOLOGICALLY_PRIORITIZED_TASKS if t in here_keep and t in other_keep]
    print(f"[Filter] sig-sample consistency >60% in both notebooks: {len(TRUE_DIFF_TASKS)} tasks")
else:
    TRUE_DIFF_TASKS=[t for t in BIOLOGICALLY_PRIORITIZED_TASKS if t in here_keep]
    print(f"[Filter] other notebook summary not found, temporarily keep local >60% tasks: {len(TRUE_DIFF_TASKS)}")

ALL_MODULE_TASKS=[t for t in ALL_MODULE_TASKS if t in TRUE_DIFF_TASKS]
MODULES={m:[t for t in ts if t in TRUE_DIFF_TASKS] for m,ts in MODULES.items()}
MODULES={m:ts for m,ts in MODULES.items() if len(ts)>0}
TASK_TO_MODULE={t:m for m,ts in MODULES.items() for t in ts}
MODULE_COLORS={m:c for m,c in MODULE_COLORS.items() if m in MODULES}
KEY_TASKS=[t for t in KEY_TASKS if t in TRUE_DIFF_TASKS]
print("[Final tasks]")
for t in TRUE_DIFF_TASKS: print("  -", t)

# 绘图
pivot_rbc=df_ps.pivot_table(index="task",columns="sample",values="rbc",aggfunc="first")
pivot_p=df_ps.pivot_table(index="task",columns="sample",values="p_value",aggfunc="first")
task_order=[t for m in MODULES for t in MODULES[m] if t in pivot_rbc.index]
pivot_rbc=pivot_rbc.loc[task_order]; pivot_p=pivot_p.reindex(pivot_rbc.index)

TASK_TO_LABEL_ID = _refresh_task_label_ids() if '_refresh_task_label_ids' in globals() else {_clean_spaces(t): i + 1 for i, t in enumerate(globals().get('TASKS_TO_LABEL', task_order))}
if '_task_number_label' not in globals():
    def _task_number_label(t):
        label_id = TASK_TO_LABEL_ID.get(_clean_spaces(t))
        return str(label_id) if label_id is not None else _short_task(str(t), 45)

_task_color={t: MODULE_COLORS[TASK_TO_MODULE[t]] for t in task_order if t in TASK_TO_MODULE}
fig=plt.figure(figsize=(14.8,13)); gs=GridSpec(1,2,width_ratios=[4.2,1.35],wspace=0.06)
ax1=fig.add_subplot(gs[0])
data_mat=pivot_rbc.values.astype(float); p_mat=pivot_p.values.astype(float)
vmax=max(0.3,np.nanpercentile(np.abs(data_mat),95))
im=ax1.imshow(data_mat,aspect="auto",cmap="RdBu_r",vmin=-vmax,vmax=vmax,interpolation="nearest")
for i in range(data_mat.shape[0]):
    for j in range(data_mat.shape[1]):
        pv=p_mat[i,j]
        if np.isnan(pv): continue
        if pv<0.001: star="***"
        elif pv<0.01: star="**"
        elif pv<0.05: star="*"
        else: continue
        fc="white" if abs(data_mat[i,j])>vmax*0.55 else "black"
        ax1.text(j,i,star,ha="center",va="center",fontsize=17.3,fontweight="bold",color=fc)
ax1.set_xticks(range(len(pivot_rbc.columns))); ax1.set_xticklabels(pivot_rbc.columns,rotation=45,ha="right",fontsize=21.3)
ax1.set_yticks(range(len(task_order))); ax1.set_yticklabels([_task_number_label(t) for t in task_order],fontsize=24)
ax1.set_ylabel('Pathway ID', fontsize=24, labelpad=10)
for i,t in enumerate(task_order):
    ax1.add_patch(plt.Rectangle((-0.8,i-0.5),0.4,1,color=_task_color.get(t,"#bdbdbd"),clip_on=False))
ax1.set_title("Rank-biserial per sample (* p<0.05  ** p<0.01  *** p<0.001)",fontsize=26.7,pad=12)
ax1.set_xlabel("Sample",fontsize=24)
cbar=fig.colorbar(im,ax=ax1,shrink=0.5,pad=0.02)
cbar.set_label("Rank-biserial (Dev vs Normal)",fontsize=21.3)
cbar.ax.tick_params(labelsize=17.3)

ax2=fig.add_subplot(gs[1])
con_vals=[df_con[df_con["task"]==t].iloc[0]["consistency_rate"] if len(df_con[df_con["task"]==t])>0 else 0 for t in task_order]
con_colors=[_task_color.get(t, "#bdbdbd") for t in task_order]
bars=ax2.barh(range(len(con_vals)), con_vals, color=con_colors, edgecolor=con_colors, linewidth=0.6, height=0.84, alpha=1.0)
for _bar, _c in zip(bars, con_colors):
    _bar.set_facecolor(_c)
    _bar.set_edgecolor(_c)
    _bar.set_alpha(1.0)
ax2.set_xlim(0,1.15); ax2.set_ylim(-0.5,len(con_vals)-0.5); ax2.invert_yaxis(); ax2.set_yticks([])
ax2.axvline(0.6,color="#999",ls="--",lw=0.8); ax2.set_xlabel("Consistency of significant samples",fontsize=21.3)
ax2.tick_params(axis="x", labelsize=17.3)
for i,v in enumerate(con_vals): ax2.text(v+0.02,i,f"{v:.0%}",va="center",fontsize=24,color="#222222")
handles=[mpatches.Patch(facecolor=c, edgecolor="none", label=m) for m,c in MODULE_COLORS.items()]
fig.legend(handles=handles,loc="lower center",ncol=3,fontsize=21.3,frameon=False,bbox_to_anchor=(0.45,-0.02))
fig.suptitle("Analysis 1: Per-sample direction consistency",fontsize=29.3,fontweight="bold",y=1.01)
plt.savefig(os.path.join(OUT_BASE,"1_per_sample_consistency_heatmap.png"),dpi=300)
plt.savefig(os.path.join(OUT_BASE,"1_per_sample_consistency_heatmap.pdf"))
plt.show(); plt.close()
print("[Saved] Analysis 1")


### 分析2：细胞类型拆分（全部细胞类型 × 全部模块任务，celltype_3 和 celltype_4 各一版）

In [ ]:
# 分别用 celltype_3_ZZM 和 celltype_4_ZZM 各绘制一遍，展示全部细胞类型
# 规则：
# 1) Epithelial/Malignant 相关细胞只用 around 数据
# 2) 其它细胞类型只用 TLS core 数据
# 3) 最终在同一张图里拼接，中间留空隙分开 core 组和 around 组
for CT_COL, ct_label in [("celltype_3_ZZM", "celltype_3"), ("celltype_4_ZZM", "celltype_4")]:
    print(f"\n{'='*50}")
    print(f"[Info] 使用 {CT_COL}")
    obs2 = adata.obs[[SAMPLE_COL, TLS_ID_COL, TLS_CLASS_COL, AROUND_COL, CT_COL]].copy()
    obs2[TLS_ID_COL] = obs2[TLS_ID_COL].astype(str)
    obs2[AROUND_COL] = obs2[AROUND_COL].astype(str)
    obs2[CT_COL] = obs2[CT_COL].astype(str)
    obs2["cls"] = obs2[TLS_CLASS_COL].map(_norm_class)
    obs2["around_cid"] = obs2[AROUND_COL].map(_around_to_core_id)

    core2 = obs2[obs2[TLS_ID_COL].map(_is_valid_id) & obs2["cls"].isin(NORMAL_CLASSES | {DEVIATING_CLASS})]
    tls2cls2 = core2.groupby(TLS_ID_COL, observed=True)["cls"].agg(lambda x: x.value_counts().index[0]).to_dict()
    tls2grp2 = {k: _get_tls_group(v) for k, v in tls2cls2.items()}
    sel2 = set(tls2grp2.keys())

    def _use_around_scope(ct_name: str) -> bool:
        s = str(ct_name).strip()
        low = s.lower()
        if "malignant" in low:
            return True
        if "epithelial" in low:
            return True
        if CT_COL == "celltype_4_ZZM":
            epi_keywords = ["at1", "at2", "club", "multiciliated", "neuroendocrine"]
            return any(k in low for k in epi_keywords)
        return False

    # core-only 数据：只保留真正的 TLS core 细胞
    core_keep = obs2[TLS_ID_COL].isin(sel2)
    o_core = obs2.loc[core_keep].copy()
    o_core["tls_link"] = o_core[TLS_ID_COL]
    o_core["tls_group"] = o_core["tls_link"].map(tls2grp2)
    o_core = o_core[o_core["tls_group"].isin(["Normal", "Deviating"])].copy()

    # around-only 数据：排除 core，只保留 around 区域挂到某个 TLS 的细胞
    around_keep = (~obs2[TLS_ID_COL].isin(sel2)) & obs2["around_cid"].isin(sel2)
    o_around = obs2.loc[around_keep].copy()
    o_around["tls_link"] = o_around["around_cid"]
    o_around["tls_group"] = o_around["tls_link"].map(tls2grp2)
    o_around = o_around[o_around["tls_group"].isin(["Normal", "Deviating"])].copy()

    X_core = adata[o_core.index].X
    if issparse(X_core):
        X_core = X_core.toarray()
    X_core = np.asarray(X_core, dtype=np.float32)

    X_around = adata[o_around.index].X
    if issparse(X_around):
        X_around = X_around.toarray()
    X_around = np.asarray(X_around, dtype=np.float32)

    all_cts = sorted(set(o_core[CT_COL].tolist()) | set(o_around[CT_COL].tolist()))
    core_cts = sorted([ct for ct in all_cts if not _use_around_scope(ct)])
    around_cts = sorted([ct for ct in all_cts if _use_around_scope(ct)])
    print(f"  Core-only cell types: {len(core_cts)}")
    print(f"  Around-only cell types: {len(around_cts)}")
    print(f"  Around-only labels: {around_cts}")

    ct_res = []

    def _collect_scope_results(o_sub, X_sub, ct_list, scope_label):
        for ct in ct_list:
            cm = o_sub[CT_COL].values == ct
            if cm.sum() == 0:
                continue
            cX = X_sub[cm]
            tids = o_sub.loc[cm, "tls_link"].values
            utls = np.unique(tids)
            ts = {}
            for tid in utls:
                tm = tids == tid
                if tm.sum() < 3:
                    continue
                ts[tid] = cX[tm].mean(axis=0)
            if len(ts) < 4:
                continue
            ntls = [t for t in ts if tls2grp2.get(t) == "Normal"]
            dtls = [t for t in ts if tls2grp2.get(t) == "Deviating"]
            if len(ntls) < 2 or len(dtls) < 2:
                continue
            for tn in ALL_MODULE_TASKS:
                j = task_names.index(tn)
                xn = np.array([ts[t][j] for t in ntls])
                xd = np.array([ts[t][j] for t in dtls])
                xn, xd = xn[np.isfinite(xn)], xd[np.isfinite(xd)]
                if xn.size < 2 or xd.size < 2:
                    continue
                try:
                    u, p = mannwhitneyu(xd, xn, alternative="two-sided", method="asymptotic")
                    rbc = (2 * u / (xd.size * xn.size)) - 1
                except Exception:
                    rbc, p = 0, 1
                ct_res.append({
                    "celltype": ct,
                    "task": tn,
                    "module": TASK_TO_MODULE[tn],
                    "rbc": rbc,
                    "p_value": p,
                    "sig": p < 0.05,
                    "plot_scope": scope_label,
                })

    _collect_scope_results(o_core, X_core, core_cts, "core")
    _collect_scope_results(o_around, X_around, around_cts, "around")

    df_ct = pd.DataFrame(ct_res)
    df_ct.to_csv(os.path.join(OUT_BASE, f"2_celltype_decomposition_{ct_label}.csv"), index=False)

    if df_ct.empty:
        print(f"[WARN] {ct_label} 没有可绘制结果")
        continue

    task_order = [t for m in MODULES for t in MODULES[m] if t in df_ct["task"].unique()]
    TASK_TO_LABEL_ID = _refresh_task_label_ids() if '_refresh_task_label_ids' in globals() else {_clean_spaces(t): i + 1 for i, t in enumerate(globals().get('TASKS_TO_LABEL', task_order))}
    if '_task_number_label' not in globals():
        def _task_number_label(t):
            label_id = TASK_TO_LABEL_ID.get(_clean_spaces(t))
            return str(label_id) if label_id is not None else _short_task(str(t), 42)
    core_ct_order = [ct for ct in core_cts if ct in df_ct[df_ct["plot_scope"] == "core"]["celltype"].unique()]
    around_ct_order = [ct for ct in around_cts if ct in df_ct[df_ct["plot_scope"] == "around"]["celltype"].unique()]
    ct_order = core_ct_order + around_ct_order

    x_pos = {}
    x_cursor = 0.0
    for ct in core_ct_order:
        x_pos[ct] = x_cursor
        x_cursor += 1.0
    if core_ct_order and around_ct_order:
        gap_left = x_cursor - 0.5
        x_cursor += 1.8
    else:
        gap_left = None
    for ct in around_ct_order:
        x_pos[ct] = x_cursor
        x_cursor += 1.0

    _task_color = {t: MODULE_COLORS[TASK_TO_MODULE[t]] for t in task_order if t in TASK_TO_MODULE}
    fig = plt.figure(figsize=(max(x_cursor * 0.72 + 2.8, 13.8), len(task_order) * 0.55 + 3.3))
    gs = GridSpec(1, 2, width_ratios=[0.45, max(x_cursor * 0.72, 11.0)], wspace=0.015)
    ax0 = fig.add_subplot(gs[0])
    ax = fig.add_subplot(gs[1])

    for i, t in enumerate(task_order):
        for ct in ct_order:
            row = df_ct[(df_ct["task"] == t) & (df_ct["celltype"] == ct)]
            if len(row) == 0:
                continue
            r = row.iloc[0]
            sz = abs(r["rbc"]) * 400 + 15
            cl = "#e41a1c" if r["rbc"] > 0 else "#377eb8"
            al = 0.9 if r["sig"] else 0.2
            ec = "black" if r["sig"] else "none"
            ax.scatter(x_pos[ct], i, s=sz, c=cl, alpha=al, edgecolors=ec,
                       linewidths=1 if r["sig"] else 0, zorder=3)

    ax.set_xticks([x_pos[ct] for ct in ct_order])
    ax.set_xticklabels(ct_order, rotation=65, ha="right", fontsize=25.3)
    ax.set_yticks(range(len(task_order)))
    ax.set_yticklabels([_task_number_label(t) for t in task_order], fontsize=25.3)
    ax.set_ylabel('Pathway ID', fontsize=25.3, labelpad=8)

    ax0.set_facecolor("white")
    ax0.set_xlim(0, 1)
    ax0.set_ylim(-0.5, len(task_order) - 0.5)
    ax0.invert_yaxis()
    ax0.set_xticks([])
    ax0.set_yticks([])
    for i, t in enumerate(task_order):
        ax0.add_patch(mpatches.Rectangle(
            (0.0, i - 0.5), 1.0, 1.0,
            facecolor=_task_color.get(t, "#bdbdbd"),
            edgecolor="none", linewidth=0, zorder=5,
        ))
    for spine in ax0.spines.values():
        spine.set_visible(False)

    ax.set_xlim(-0.5, x_cursor - 0.5)
    ax.set_ylim(-0.5, len(task_order) - 0.5)
    ax.grid(True, ls=":", alpha=0.3)
    ax.invert_yaxis()

    if gap_left is not None:
        ax.axvline(gap_left + 0.9, color="#9a9a9a", lw=1.2, ls="--", alpha=0.9)
        core_center = np.mean([x_pos[c] for c in core_ct_order]) if core_ct_order else 0
        around_center = np.mean([x_pos[c] for c in around_ct_order]) if around_ct_order else 0
        ax.text(core_center, 1.02, "TLS core only", transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=22.0, color="#444444")
        ax.text(around_center, 1.02, "TLS around only", transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=22.0, color="#444444")

    for sv, sl in [(0.15, "0.15"), (0.30, "0.30"), (0.50, "0.50")]:
        ax.scatter([], [], s=sv * 400 + 15, c="grey", alpha=0.6, label=f"|rbc|={sl}")
    ax.scatter([], [], s=100, c="#e41a1c", label="Deviating ↑")
    ax.scatter([], [], s=100, c="#377eb8", label="Normal ↑")
    ax.scatter([], [], s=100, c="grey", edgecolors="black", linewidths=1.2, label="p < 0.05")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=20.0, frameon=False)
    ax.set_title(
        f"Analysis 2: Cell type drivers ({ct_label})\n"
        f"core  |around red=Deviating↑  blue=Normal↑  black border=p<0.05",
        fontsize=29.0, fontweight="bold", pad=18
    )
    plt.savefig(os.path.join(OUT_BASE, f"2_celltype_bubble_{ct_label}.png"), dpi=300)
    plt.savefig(os.path.join(OUT_BASE, f"2_celltype_bubble_{ct_label}.pdf"))
    plt.show(); plt.close()
    print(f"[Saved] 2_celltype_bubble_{ct_label}")


### 分析3：空间梯度 Core → Around → Outside

In [ ]:
import textwrap

obs3=adata.obs[[SAMPLE_COL,TLS_ID_COL,TLS_CLASS_COL,AROUND_COL]].copy()
obs3[TLS_ID_COL]=obs3[TLS_ID_COL].astype(str); obs3[AROUND_COL]=obs3[AROUND_COL].astype(str)
obs3["cls"]=obs3[TLS_CLASS_COL].map(_norm_class); obs3["around_cid"]=obs3[AROUND_COL].map(_around_to_core_id)

is_core=obs3[TLS_ID_COL].map(_is_valid_id)
is_around=(~is_core)&obs3["around_cid"].map(lambda x:_is_valid_id(x) if x else False)
obs3["zone"]="Outside"; obs3.loc[is_core,"zone"]="Core"; obs3.loc[is_around,"zone"]="Around"

ct2c=obs3.loc[is_core].groupby(TLS_ID_COL,observed=True)["cls"].agg(lambda x:x.value_counts().index[0]).to_dict()
ct2g={k:_get_tls_group(v) for k,v in ct2c.items()}
obs3["tls_link3"]="<NA>"; obs3.loc[is_core,"tls_link3"]=obs3.loc[is_core,TLS_ID_COL]; obs3.loc[is_around,"tls_link3"]=obs3.loc[is_around,"around_cid"]
obs3["tls_grp3"]=obs3["tls_link3"].map(lambda x:ct2g.get(x,"<NA>"))
stls=set(obs3.loc[obs3["tls_grp3"].isin(["Normal","Deviating"]),SAMPLE_COL].unique())
obs3.loc[(obs3["zone"]=="Outside")&(obs3[SAMPLE_COL].isin(stls)),"tls_grp3"]="Background"

X3=adata.X
if issparse(X3): X3=X3.toarray()
X3=np.asarray(X3,dtype=np.float32)

def _wrap_task_title(task_name, short_width=32, wrap_width=16, max_lines=2):
    txt = _short_task(task_name, short_width)
    lines = textwrap.wrap(txt, width=wrap_width, break_long_words=False, break_on_hyphens=False)
    if len(lines) > max_lines:
        lines = lines[:max_lines]
        if not lines[-1].endswith('...'):
            lines[-1] = lines[-1].rstrip('.') + '...'
    return "\n".join(lines)

grad_rows=[]
for smp in sorted(stls):
    sm=obs3[SAMPLE_COL].values==smp
    for zone in ["Core","Around","Outside"]:
        zm=obs3["zone"].values==zone
        for gl in (["Normal","Deviating"] if zone!="Outside" else ["Background"]):
            cm=sm&zm&(obs3["tls_grp3"].values==gl); n=int(cm.sum())
            if n<5: continue
            for t in KEY_TASKS:
                j=task_names.index(t); vals=X3[cm,j]; vals=vals[np.isfinite(vals)]
                if vals.size<5: continue
                grad_rows.append({"sample":smp,"zone":zone,"tls_group":gl,"task":t,"mean_score":float(np.mean(vals)),"n_cells":n})
df_grad=pd.DataFrame(grad_rows)
df_grad.to_csv(os.path.join(OUT_BASE,"3_spatial_gradient.csv"),index=False)

n_tasks = len(KEY_TASKS)
ncols = min(4, n_tasks)
nrows = int(np.ceil(n_tasks / ncols))
fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*4.1, nrows*3.7),sharey=False)
axes=np.atleast_1d(axes).ravel()
for ai,t in enumerate(KEY_TASKS):
    ax=axes[ai]; sub=df_grad[df_grad["task"]==t]
    for gl,c in [("Normal","#377eb8"),("Deviating","#e41a1c")]:
        means,sems=[],[]
        for z in ["Core","Around","Outside"]:
            zs=sub[(sub["zone"]==z)&(sub["tls_group"].isin([gl,"Background"]))]
            if len(zs)>0:
                means.append(np.mean(zs["mean_score"]))
                sems.append(np.std(zs["mean_score"])/np.sqrt(len(zs)) if len(zs)>1 else 0)
            else:
                means.append(np.nan); sems.append(0)
        ax.errorbar([0,1,2],means,yerr=sems,marker="o",ms=7,color=c,lw=0.36,capsize=4,capthick=0.24,label=gl,alpha=0.85)
    ax.set_xticks([0,1,2]); ax.set_xticklabels(["Core","Around","Outside"],fontsize=18)
    ax.tick_params(axis='y', labelsize=16)
    ax.set_title(_wrap_task_title(t, 32, 16),fontsize=18,fontweight="bold",pad=9)
    if ai % ncols == 0:
        ax.set_ylabel("Mean metabolic score",fontsize=18)
    else:
        ax.set_ylabel("")
    ax.grid(True,axis="y",ls=":",alpha=0.3)
for ax in axes[n_tasks:]:
    ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.985, 0.985), fontsize=18, frameon=False)
fig.suptitle("Analysis 3: Spatial gradient — Core → Around → Outside",fontsize=24,fontweight="bold",y=0.995)
plt.tight_layout(rect=(0.02, 0.02, 0.96, 0.95))
plt.savefig(os.path.join(OUT_BASE,"3_spatial_gradient.png"),dpi=300)
plt.savefig(os.path.join(OUT_BASE,"3_spatial_gradient.pdf"))
plt.show(); plt.close()
print("[Saved] Analysis 3")


In [ ]:
# ============================================================
# 分析6: Deviating TLS vs Normal TLS — 特定细胞类型基因表达差异
# ============================================================
# 基因: EPAS1, HSPB1, HSPA1A
# 细胞类型使用: celltype_3_ZZM
# 新规则:
#   1) 免疫细胞只用 Core only
#   2) 非免疫细胞用 Core + Around
#   3) 最终拼成一张图
# 方法: TLS-level pseudobulk (mean) + Mann-Whitney U test
# ============================================================

import os, warnings
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse
from scipy.stats import mannwhitneyu
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'axes.linewidth': 1.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SCCF_FULL_PATH6 = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/2026_3_24/slide-tag_lung_scCellFie.h5ad"
OUT_DIR6 = "/data/beifen/zhongmin/slide-tag/代谢/scCellFie/优化比较代谢差异/不分样本比较_伪bulk/基因表达差异"
os.makedirs(OUT_DIR6, exist_ok=True)

GENES6 = ["EPAS1"]
CELLTYPE_COL6 = "celltype_3_ZZM"
CELL_TYPES6 = ["B cells","T cells","Macrophages","Fibroblasts","Endothelial cells","Malignant cells"]
#["B cells","Plasma cells", "T cells","Treg", "CD4 Trm","CD8 Trm","Tfh","NK cells","Macrophages","DC","Mast cells","FDC","Fibroblasts", "Endothelial cells" ,"Pericyte","SMC","Multiciliated","Epithelial cells","Malignant cells"]
IMMUNE_CELL_TYPES6 = [
    "B cells", "Plasma cells", "T cells", "Treg", "CD4 Trm", "CD8 Trm", "Tfh",
    "NK cells", "Macrophages", "DC", "Mast cells", "FDC"
]
NONIMMUNE_CELL_TYPES6 = [ct for ct in CELL_TYPES6 if ct not in IMMUNE_CELL_TYPES6]
SCOPE_MAP6 = {ct: ("Core only" if ct in IMMUNE_CELL_TYPES6 else "Core plus Around") for ct in CELL_TYPES6}
MIN_CELLS_TLS6 = 3
PAL6 = {"Normal": "#4393C3", "Deviating": "#D6604D"}

print("=" * 60)
print("分析6: Gene expression — Normal TLS vs Deviating TLS")
print("=" * 60)
print("\n[1/6] 加载基因表达 (scCellFie .raw 层) ...")
adata_full6 = sc.read_h5ad(SCCF_FULL_PATH6)

available6 = [g for g in GENES6 if g in adata_full6.raw.var_names]
missing6 = [g for g in GENES6 if g not in adata_full6.raw.var_names]
if missing6:
    print(f"  [WARN] 以下基因不在数据中: {missing6}")
GENES6 = available6
assert len(GENES6) > 0, "没有可用基因！请检查 GENES6 列表。"
print(f"  可用基因: {GENES6}")

raw_X6 = adata_full6.raw[:, GENES6].X
if issparse(raw_X6):
    raw_X6 = raw_X6.toarray()
gene_df6 = pd.DataFrame(np.asarray(raw_X6, dtype=np.float32), index=adata_full6.obs_names, columns=GENES6)
del adata_full6, raw_X6

print("[2/6] 构建分析数据 ...")
obs6 = adata.obs[[TLS_ID_COL, TLS_CLASS_COL, AROUND_COL, CELLTYPE_COL6]].copy()
df6_all = obs6.join(gene_df6, how="inner")
del gene_df6

df6_all["tls_class"] = df6_all[TLS_CLASS_COL].map(_norm_class)
df6_all["tls_id"] = df6_all[TLS_ID_COL].astype(str)
df6_all["around_id"] = df6_all[AROUND_COL].astype(str)
df6_all["around_core_id"] = df6_all["around_id"].map(_around_to_core_id)

core_valid6 = (
    df6_all["tls_id"].map(_is_valid_id)
    & df6_all["tls_class"].isin(NORMAL_CLASSES | {DEVIATING_CLASS})
)
tls2cls6 = (
    df6_all.loc[core_valid6]
    .groupby("tls_id", observed=True)["tls_class"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)
tls2grp6 = {k: ("Normal" if v in NORMAL_CLASSES else "Deviating") for k, v in tls2cls6.items()}
sel_tls6 = set(tls2grp6.keys())

n_norm_tls = sum(1 for v in tls2grp6.values() if v == "Normal")
n_dev_tls = sum(1 for v in tls2grp6.values() if v == "Deviating")
print(f"  有效 TLS: Normal={n_norm_tls}, Deviating={n_dev_tls}")

print("[3/6] 构建 Core only / Core + Around 数据 ...")
mask_core = df6_all["tls_id"].isin(sel_tls6)
df6_core = df6_all.loc[mask_core].copy()
df6_core["tls_link"] = df6_core["tls_id"]
df6_core["group"] = df6_core["tls_link"].map(tls2grp6)
df6_core = df6_core[df6_core["group"].isin(["Normal", "Deviating"])].copy()
df6_core = df6_core[df6_core[CELLTYPE_COL6].isin(CELL_TYPES6)].copy()

ca_core = df6_all.loc[mask_core].copy()
ca_core["tls_link"] = ca_core["tls_id"]
mask_around = (~df6_all["tls_id"].isin(sel_tls6)) & df6_all["around_core_id"].isin(sel_tls6)
ca_around = df6_all.loc[mask_around].copy()
ca_around["tls_link"] = ca_around["around_core_id"]
df6_ca = pd.concat([ca_core, ca_around], axis=0)
df6_ca["group"] = df6_ca["tls_link"].map(tls2grp6)
df6_ca = df6_ca[df6_ca["group"].isin(["Normal", "Deviating"])].copy()
df6_ca = df6_ca[df6_ca[CELLTYPE_COL6].isin(CELL_TYPES6)].copy()

for scope_name, scope_df in [("Core only", df6_core), ("Core + Around", df6_ca)]:
    print(f"\n  [{scope_name}]  细胞总数: {len(scope_df)}")
    for ct in CELL_TYPES6:
        nn = ((scope_df[CELLTYPE_COL6] == ct) & (scope_df["group"] == "Normal")).sum()
        nd = ((scope_df[CELLTYPE_COL6] == ct) & (scope_df["group"] == "Deviating")).sum()
        print(f"    {ct}: Normal={nn}, Deviating={nd}")


def _sig_star6(p):
    if pd.isna(p) or p >= 0.05:
        return "ns"
    if p >= 0.01:
        return "*"
    if p >= 0.001:
        return "**"
    return "***"


def run_pseudobulk_and_stats(scope_df, genes, cell_types, min_cells):
    pb_rows = []
    for (ct, tid), sub in scope_df.groupby([CELLTYPE_COL6, "tls_link"], observed=True):
        if len(sub) < min_cells:
            continue
        row = {
            "celltype": ct,
            "tls_id": tid,
            "group": sub["group"].iloc[0],
            "n_cells": len(sub),
        }
        for g in genes:
            row[g] = sub[g].mean()
        pb_rows.append(row)
    pb = pd.DataFrame(pb_rows)

    stat_rows = []
    for ct in cell_types:
        for gene in genes:
            if pb.empty:
                xn = np.array([])
                xd = np.array([])
            else:
                xn = pb.loc[(pb["celltype"] == ct) & (pb["group"] == "Normal"), gene].dropna().values
                xd = pb.loc[(pb["celltype"] == ct) & (pb["group"] == "Deviating"), gene].dropna().values
            if len(xn) >= 3 and len(xd) >= 3:
                u_val, p_val = mannwhitneyu(xd, xn, alternative="two-sided")
                rbc = (2 * u_val / (len(xd) * len(xn))) - 1
            else:
                u_val, p_val, rbc = np.nan, np.nan, np.nan
            stat_rows.append(dict(
                celltype=ct,
                gene=gene,
                n_normal_tls=len(xn),
                n_deviating_tls=len(xd),
                mean_normal=np.nanmean(xn) if len(xn) else np.nan,
                mean_deviating=np.nanmean(xd) if len(xd) else np.nan,
                U_stat=u_val,
                p_value=p_val,
                rank_biserial=rbc,
            ))
    df_stat = pd.DataFrame(stat_rows)
    df_stat["sig"] = df_stat["p_value"].map(_sig_star6)
    return pb, df_stat


print("[4/6] 计算 Core only / Core + Around pseudobulk ...")
pb_core, stat_core = run_pseudobulk_and_stats(df6_core, GENES6, CELL_TYPES6, MIN_CELLS_TLS6)
pb_ca, stat_ca = run_pseudobulk_and_stats(df6_ca, GENES6, CELL_TYPES6, MIN_CELLS_TLS6)

pb_mixed_parts = []
stat_mixed_parts = []
for ct in CELL_TYPES6:
    use_scope = SCOPE_MAP6[ct]
    if use_scope == "Core only":
        pb_src, stat_src = pb_core, stat_core
    else:
        pb_src, stat_src = pb_ca, stat_ca

    if not pb_src.empty:
        pb_sub = pb_src[pb_src["celltype"] == ct].copy()
        pb_sub["scope_used"] = use_scope
        pb_mixed_parts.append(pb_sub)

    stat_sub = stat_src[stat_src["celltype"] == ct].copy()
    stat_sub["scope_used"] = use_scope
    stat_mixed_parts.append(stat_sub)

pb_mixed = pd.concat(pb_mixed_parts, ignore_index=True) if len(pb_mixed_parts) else pd.DataFrame()
stat_mixed = pd.concat(stat_mixed_parts, ignore_index=True) if len(stat_mixed_parts) else pd.DataFrame()

print("\n  最终混合策略:")
for ct in CELL_TYPES6:
    print(f"    {ct}: {SCOPE_MAP6[ct]}")

print("\n  最终用于绘图的 TLS pseudobulk 数:")
for ct in CELL_TYPES6:
    nn = ((pb_mixed["celltype"] == ct) & (pb_mixed["group"] == "Normal")).sum() if not pb_mixed.empty else 0
    nd = ((pb_mixed["celltype"] == ct) & (pb_mixed["group"] == "Deviating")).sum() if not pb_mixed.empty else 0
    print(f"    {ct}: Normal TLS={nn}, Deviating TLS={nd}, scope={SCOPE_MAP6[ct]}")

print("\n  Mann-Whitney U test 结果:")
print("  " + "-" * 96)
print(stat_mixed[["celltype", "scope_used", "gene", "n_normal_tls", "n_deviating_tls", "mean_normal", "mean_deviating", "p_value", "sig"]].to_string(index=False))
print("  " + "-" * 96)


def plot_box_strip_mixed(pb, df_stat, genes, cell_types, scope_map, pal, out_dir):
    plot_cell_types = [ct for ct in cell_types if (not pb.empty) and (pb["celltype"] == ct).any()]
    if len(plot_cell_types) == 0:
        raise ValueError("No cell types with pseudobulk results available for plotting.")

    n_g = len(genes)
    rng = np.random.default_rng(2026)
    fig_h = max(7.2, 0.66 * len(plot_cell_types) * n_g + 1.6)
    fig, axes = plt.subplots(n_g, 1, figsize=(11.8, fig_h), gridspec_kw={"hspace": 0.34})
    if n_g == 1:
        axes = [axes]

    for gi, gene in enumerate(genes):
        ax = axes[gi]
        gene_vals = pb[gene].dropna().values if not pb.empty else np.array([])
        gene_min = np.min(gene_vals) if len(gene_vals) else 0.0
        gene_max = np.max(gene_vals) if len(gene_vals) else 1.0
        x_span = gene_max - gene_min
        if x_span == 0:
            x_span = 1.0

        dodge = 0.18
        widths = 0.26
        y_pos = np.arange(len(plot_cell_types), dtype=float)

        for yi, ct in enumerate(plot_cell_types):
            if scope_map[ct] != "Core only":
                ax.axhspan(yi - 0.5, yi + 0.5, color="#fbf7f1", zorder=0)

            for grp, offset in [("Normal", -dodge), ("Deviating", dodge)]:
                vals = pb.loc[(pb["celltype"] == ct) & (pb["group"] == grp), gene].dropna().values if not pb.empty else np.array([])
                if len(vals) == 0:
                    continue
                y_center = yi + offset
                ax.boxplot(
                    vals,
                    positions=[y_center],
                    vert=False,
                    widths=widths,
                    patch_artist=True,
                    showfliers=False,
                    zorder=3,
                    boxprops=dict(facecolor=pal[grp], alpha=0.42, edgecolor="#333333", linewidth=0.85),
                    whiskerprops=dict(color="#555555", linewidth=0.8),
                    capprops=dict(color="#555555", linewidth=0.8),
                    medianprops=dict(color="#111111", linewidth=1.4),
                )
                jitter = rng.uniform(-0.055, 0.055, size=len(vals))
                ax.scatter(
                    vals,
                    np.full(len(vals), y_center) + jitter,
                    s=34,
                    c=pal[grp],
                    alpha=0.82,
                    edgecolors="white",
                    linewidths=0.45,
                    zorder=5,
                )

            row = df_stat[(df_stat["celltype"] == ct) & (df_stat["gene"] == gene)]
            if row.empty:
                continue
            p_val = row["p_value"].iloc[0]
            sig_txt = row["sig"].iloc[0]
            all_v = pb.loc[pb["celltype"] == ct, gene].dropna().values if not pb.empty else np.array([])
            if len(all_v) == 0:
                continue

            local_max = np.max(all_v)
            x0 = local_max + x_span * 0.030
            x1 = x0 + x_span * 0.018
            y0 = yi - dodge
            y1 = yi + dodge
            ax.plot([x0, x1, x1, x0], [y0, y0, y1, y1], lw=1.0, c="#222222", clip_on=False, zorder=6)

            if pd.isna(p_val):
                label_txt = "n/a"
            elif sig_txt == "ns":
                label_txt = "ns"
            else:
                p_str = f"p={p_val:.1e}" if p_val < 0.001 else f"p={p_val:.3f}"
                label_txt = f"{sig_txt}  {p_str}"
            ax.text(x1 + x_span * 0.010, yi, label_txt, ha="left", va="center", fontsize=10.5, fontweight="bold", color="#1f1f1f")

        ylabels = []
        for ct in plot_cell_types:
            scope_txt = "core" if scope_map[ct] == "Core only" else "core + around"
            ylabels.append(f"{ct}   [{scope_txt}]")
        ax.set_yticks(y_pos)
        ax.set_yticklabels(ylabels, fontsize=12)
        ax.invert_yaxis()
        ax.set_xlabel("Expression (raw counts)", fontsize=14)
        ax.set_title(gene, fontsize=19, fontweight="bold", fontstyle="italic", pad=10)
        ax.tick_params(axis="x", labelsize=12, length=4, width=0.8)
        ax.tick_params(axis="y", length=0)
        ax.grid(axis="x", linestyle=":", linewidth=0.8, alpha=0.35, color="#9aa5b1")
        ax.set_axisbelow(True)
        ax.set_xlim(gene_min - x_span * 0.04, gene_max + x_span * 0.38)
        ax.spines["left"].set_linewidth(0.8)
        ax.spines["bottom"].set_linewidth(0.8)

    legend_handles = [
        mpatches.Patch(facecolor=pal["Normal"], alpha=0.55, edgecolor="#333333", linewidth=0.6, label="Normal TLS (Conforming + Mature)"),
        mpatches.Patch(facecolor=pal["Deviating"], alpha=0.55, edgecolor="#333333", linewidth=0.6, label="Deviating TLS"),
    ]
    fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=2, fontsize=12, frameon=False)
    fig.suptitle("Gene expression in TLS: Normal vs Deviating", fontsize=20, fontweight="bold", y=1.08)
    fig.text(
        0.5,
        1.02,
        "Immune cell types: core only   |   Non-immune cell types: core + around",
        ha="center",
        va="center",
        fontsize=12.5,
        color="#5f6b7a",
    )

    out_png = os.path.join(out_dir, "gene_expr_NvsD_mixed_scope.png")
    out_pdf = os.path.join(out_dir, "gene_expr_NvsD_mixed_scope.pdf")
    fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close()
    return out_png, out_pdf


print("[5/6] 保存统计表并绘图 ...")
stat_csv = os.path.join(OUT_DIR6, "gene_expr_NvsD_mixed_scope_stats.csv")
stat_mixed.to_csv(stat_csv, index=False)
print(f"\n  [Saved] {stat_csv}")

out_png, out_pdf = plot_box_strip_mixed(pb_mixed, stat_mixed, GENES6, CELL_TYPES6, SCOPE_MAP6, PAL6, OUT_DIR6)

print(f"\n{'='*60}")
print("分析6 完成! 已保存文件:")
for f in [stat_csv, out_png, out_pdf]:
    print(f"  {f}")
print(f"{'='*60}")
